# Kurs Kassası: qəbz oxuyucusu - QLoRA (Qwen2.5-1.5B), Colab T4

Valideyn kursa pul atanda rəhbərə bank SMS-i, kağız qəbzin şəkli və ya «Ayselin noyabr pulunu
köçürdüm» kimi mesaj gəlir. Kiçik model bu mətndən JSON çıxarır: məbləğ, tarix, ödəyən, kartın son
4 rəqəmi, hansı ay, şagird. Öz modelimiz ona görədir ki, mətndə ad və kart rəqəmi var: defoltda
Groq/Claude-a getmir (`RECEIPT_ALLOW_EXTERNAL_LLM=false`).

**Necə işlətmək:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. Drive icazəsi
soruşulanda təsdiqlə. Təxminən: paketlər 3 dəq, öyrətmə 10-20 dəq, qiymətləndirmə 10-15 dəq.

**Nə ölçülür:** `evals.kassa`-nın EYNİ qəbzləri (60 × 6 növ, seed 11): görülən 4 növ (240) və
öyrətməyə HEÇ düşməyən 2 növ (`ewallet`, `messenger`, 120). Müqayisə: qaydalar (92% / 50%) və
Groq (88% / 100%, serverdə həmin dəstin 60 qəbzində). Data sintetikdir (`ml/receipts/synth.py`).

**Drive-da qalır** (`MyDrive/masterai-receipts/`): adapter (~70 MB), checkpoint-lər, loss tarixçəsi,
modelin cavabları, `report.md` + `report.json`. Sessiya qırılsa **Run all** qaldığı yerdən davam edir:
hazır adapter və cavablar yenidən hesablanmır.

In [ ]:
# 1) GPU + Drive + parametrlər
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, torch
# GPU yoxdursa öyrətmə CPU-da saatlarla gedir: burada dayandır
assert torch.cuda.is_available(), "GPU YOXDUR. Runtime → Change runtime type → T4 GPU, sonra Restart session + Run all"
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/masterai-receipts"
ADAPTER = f"{DRIVE}/adapter"            # yekun LoRA adapteri (vLLM bunu yükləyir)
CKPT = f"{DRIVE}/checkpoints"           # öyrətmə qırılsa buradan davam edir
OUTPUTS = f"{DRIVE}/outputs"            # modelin eval cavabları
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RETRAIN = False                         # True: Drive-dakı adapteri saymayıb yenidən öyrət
os.makedirs(OUTPUTS, exist_ok=True)
ALREADY_TRAINED = os.path.exists(f"{ADAPTER}/adapter_config.json") and not RETRAIN
print("Adapter Drive-da var, öyrətmə atlanacaq." if ALREADY_TRAINED else "Öyrətmə işləyəcək.")

In [ ]:
# 2) Paketlər + repo. Səhv Detektivi notebook-unda Colab-da işləyən yol: ən son versiyalar
# (köhnə sabit versiyalar Colab-ın torch-u ilə toqquşur). Versiyalar hesabata yazılır.
# pip-in "cudf ... pyarrow" xəbərdarlığı zərərsizdir.
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets
!rm -rf masterai && git clone -q --depth 1 https://github.com/Alihuseyn-ibrahimov/masterai.git
import sys, importlib.metadata as md
sys.path.insert(0, "/content/masterai")
VERSIONS = {p: md.version(p) for p in ("torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate")}
print(VERSIONS)

In [ ]:
# 3) Dataset: öyrətmə seed 7 (görülən 4 növ), eval seed 11 (evals.kassa ilə eyni)
!cd /content/masterai && python -m ml.receipts.sft --per-family 400 --seed 7 --out /content/receipts
import json
from datasets import Dataset
from ml.receipts import compare

def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows, val_rows = read_jsonl("/content/receipts/train.jsonl"), read_jsonl("/content/receipts/val.jsonl")

def to_pc(rows):
    # prompt/completion formatı: loss YALNIZ JSON cavabında hesablanır, sistem promptunda yox
    return Dataset.from_list([{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in rows])

train_ds, val_ds = to_pc(train_rows), to_pc(val_rows)
parts = compare.eval_parts()
eval_prompts = compare.prompts(parts)
leak = compare.overlap_with_train(parts, {r["text"] for r in train_rows})
print(f"train {len(train_ds)}, val {len(val_ds)}, eval görülən {len(parts['seen'])}, "
      f"görülməyən {len(parts['unseen'])}, eval-da öyrətmə mətni (sızma): {leak}")
print("\nNümunə:\n" + train_rows[0]["text"] + "\n→ " + train_rows[0]["messages"][2]["content"])

In [ ]:
# 4) 4-bit baza + LoRA. LoRA çəkiləri fp32-də: fp16=True Colab-da BFloat16 GradScaler xətası verirdi
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)   # T4 bf16 dəstəkləmir
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto",
                                             dtype=torch.float16)
print(f"Baza yaddaşı: {model.get_memory_footprint() / 1e9:.2f} GB")
if ALREADY_TRAINED:
    model = PeftModel.from_pretrained(model, ADAPTER)
else:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    print("Öyrədilən çəkilərin tipi:", {str(p.dtype) for p in model.parameters() if p.requires_grad})
model.print_trainable_parameters()

In [ ]:
# 5) Öyrətmə: 1280 nümunə × 2 epoch = 160 addım. Checkpoint hər 40 addımda Drive-a.
if not ALREADY_TRAINED:
    from transformers.trainer_utils import get_last_checkpoint
    from trl import SFTConfig, SFTTrainer
    cfg = SFTConfig(
        output_dir=CKPT, num_train_epochs=2, per_device_train_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=8, max_length=512,
        logging_steps=5, eval_strategy="steps", eval_steps=40, per_device_eval_batch_size=8,
        save_strategy="steps", save_steps=40, save_total_limit=2,
        fp16=False, bf16=False, gradient_checkpointing=True, optim="paged_adamw_8bit", report_to="none")
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tok)
    labels = trainer.data_collator([trainer.train_dataset[0]])["labels"][0]
    print("loss hesablanan token:", int((labels != -100).sum()), "/", len(labels))   # yalnız JSON olmalıdır
    last = get_last_checkpoint(CKPT) if os.path.isdir(CKPT) else None
    print("Davam edilən checkpoint:", last)
    trainer.train(resume_from_checkpoint=last)
    trainer.save_model(ADAPTER)
    json.dump(trainer.state.log_history, open(f"{ADAPTER}/log_history.json", "w"))
!ls -lh {ADAPTER}

In [ ]:
# 6) Loss əyrisi
import matplotlib.pyplot as plt
path = f"{ADAPTER}/log_history.json"
if os.path.exists(path):
    h = json.load(open(path))
    tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
    ev = [(x["step"], x["eval_loss"]) for x in h if "eval_loss" in x]
    plt.figure(figsize=(8, 4)); plt.plot(*zip(*tr), label="train")
    if ev:
        plt.plot(*zip(*ev), "o-", label="val")
    plt.xlabel("addım"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.title("Qəbz oxuyucusu QLoRA"); plt.show()

In [ ]:
# 7) Eval cavabları: baza model (adaptersiz) və QLoRA, 360 qəbz, greedy. Drive-a yazılır.
import time
model.eval()
model.config.use_cache = True
tok.padding_side = "left"

def generate(batch, max_new_tokens=128):
    texts = [tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True) for r in batch]
    inputs = tok(texts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def answers(name, use_adapter, batch_size=16):
    path = f"{OUTPUTS}/{name}.jsonl"
    done = {r["text"]: r["output"] for r in read_jsonl(path)} if os.path.exists(path) else {}
    todo = [r for r in eval_prompts if r["text"] not in done]
    started = time.time()
    with open(path, "a", encoding="utf-8") as fh:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            if use_adapter:
                outs = generate(batch)
            else:
                with model.disable_adapter():
                    outs = generate(batch)
            for r, o in zip(batch, outs):
                done[r["text"]] = o
                fh.write(json.dumps({"text": r["text"], "output": o}, ensure_ascii=False) + "\n")
            print(f"{name}: {len(done)}/{len(eval_prompts)}", end="\r")
    if todo:
        print(f"{name}: {len(todo)} qəbz, {(time.time() - started) / len(todo):.2f} san/qəbz")
    return done

outputs = {"base": answers("base", use_adapter=False), "qlora": answers("qlora", use_adapter=True)}

In [ ]:
# 8) Müqayisə: qaydalar vs baza vs QLoRA (xam JSON və produksiya yolu) + Groq (əvvəlki ölçmə)
from IPython.display import Markdown, display
report = compare.compare(outputs, parts)
report["versions"], report["leak"] = VERSIONS, leak
compare.save(report, f"{DRIVE}/report.md")
display(Markdown(compare.format_report(report)))
print(f"\nSaxlanıldı: {DRIVE}/report.md və report.json")

In [ ]:
# 9) QLoRA hansı qəbzlərdə yanılır (görülməyən növlər, hər növdən 3 nümunə, xam JSON)
from ml.receipts.score import parse_outputs, _same
from ml.receipts.synth import FIELDS
shown, per_family = 0, {}
for r in parts["unseen"]:
    pred = parse_outputs([outputs["qlora"][r["text"]]])[0]
    wrong = [f for f in FIELDS if pred is None or not _same(f, r["label"][f], pred.model_dump()[f])]
    if wrong and per_family.get(r["family"], 0) < 3:
        per_family[r["family"]] = per_family.get(r["family"], 0) + 1
        shown += 1
        print(f"--- {r['family']} | səhv sahələr: {wrong}\n{r['text']}\nGÖZLƏNİLƏN: {json.dumps(r['label'], ensure_ascii=False)}"
              f"\nMODEL:      {outputs['qlora'][r['text']].strip()}\n")
print("Görülməyən növlərdə səhv yoxdur." if shown == 0 else "")

## Nəticəni göndər
`report.md` faylının məzmununu (8-ci hüceyrənin cədvəli) çata yapışdır. Adapter Drive-dadır:
vLLM ilə serve etmək üçün `ml/vllm_receipts.ipynb` (multi-LoRA, `guided_json`, sürət, MasterAI-yə qoşmaq).